# Feature Engineering Mastery

Feature engineering is where **raw data becomes predictive power**. Models can only learn from what you give them — weak features limit even the best algorithms.

## Four Types of Feature Engineering

| Type | Description |
|---|---|
| **Numeric Transformations** | Scaling, binning, log/power transforms |
| **Categorical Encoding** | One-hot, label, ordinal encoding |
| **Date Feature Extraction** | Year, month, weekday, quarter, seasonality |
| **Interaction Features** | Combining variables to capture joint effects |

## Three Reasons Feature Engineering Matters
1. **Model Performance Dependency** — features determine the upper limit of performance
2. **Domain Intelligence** — embeds business knowledge directly into the dataset
3. **Signal Extraction** — converts noisy raw inputs into structured signals

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

np.random.seed(42)

# Dataset for feature engineering demonstrations
n = 1000
df = pd.DataFrame({
    'age':         np.random.randint(18, 70, n),
    'income':      np.random.exponential(50000, n).round(2),  # right-skewed
    'score':       np.random.uniform(1, 10, n).round(2),
    'city':        np.random.choice(['Mumbai', 'Delhi', 'Bangalore', 'Chennai'], n),
    'tier':        np.random.choice(['Bronze', 'Silver', 'Gold', 'Platinum'], n),
    'signup_date': pd.date_range('2020-01-01', periods=n, freq='12h'),
    'quantity':    np.random.randint(1, 20, n),
    'price':       np.random.uniform(100, 5000, n).round(2),
    'cost':        np.random.uniform(50, 2500, n).round(2),
})

print(df.shape)
df.head()

## 1. Binning (Discretisation)

Converts continuous variables into discrete categories. Useful for:
- Capturing **non-linear relationships** (models that assume linearity)
- Aligning with **business logic** (marketing tiers, pricing brackets)

In [ ]:
# Age bins informed by business / domain logic
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 25, 35, 50, 100],
    labels=['18-25', '26-35', '36-50', '51+']
)

# Quantile-based binning for income (equal-frequency buckets)
df['income_quartile'] = pd.qcut(
    df['income'],
    q=4,
    labels=['Q1', 'Q2', 'Q3', 'Q4']
)

print("Age group distribution:")
print(df['age_group'].value_counts().sort_index())

print("\nIncome quartile distribution:")
print(df['income_quartile'].value_counts().sort_index())

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Binning — Continuous Variables Converted to Discrete Categories',
             fontsize=13, fontweight='bold')

bin_edges  = [0, 25, 35, 50, 100]
bin_colors = ['#1565C0', '#2E7D32', '#E65100', '#6A1B9A']
bin_labels = ['18-25', '26-35', '36-50', '51+']

# Panel 1 — Age histogram with bin boundaries overlaid
axes[0].hist(df['age'], bins=30, color='#90CAF9', edgecolor='white', alpha=0.85)
for edge, col in zip(bin_edges[1:-1], bin_colors):
    axes[0].axvline(edge, color=col, linewidth=2.2, linestyle='--', label=f'cut at {edge}')
axes[0].set_title('Age: Histogram with pd.cut() Boundaries')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].legend(fontsize=8)
axes[0].spines[['top', 'right']].set_visible(False)

# Panel 2 — Resulting age_group counts (equal-width bins)
age_counts = df['age_group'].value_counts().sort_index()
bars = axes[1].bar(age_counts.index, age_counts.values,
                   color=bin_colors, edgecolor='white', linewidth=1.5, alpha=0.9)
for bar, count in zip(bars, age_counts.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 3, str(count),
                 ha='center', fontweight='bold')
axes[1].set_title('After pd.cut() — Age Groups (Domain Bins)')
axes[1].set_ylabel('Count')
axes[1].spines[['top', 'right']].set_visible(False)

# Panel 3 — Income with quantile cut-points overlaid
q_cuts  = df['income'].quantile([0.25, 0.5, 0.75])
q_cols  = ['#1565C0', '#E65100', '#6A1B9A']
q_names = ['Q1 (25th pct)', 'Median (50th)', 'Q3 (75th pct)']
axes[2].hist(df['income'], bins=50, color='#FFB74D', edgecolor='white', alpha=0.85)
for q, col, name in zip(q_cuts, q_cols, q_names):
    axes[2].axvline(q, color=col, linewidth=2, linestyle='--',
                    label=f'{name}\n= {q:,.0f}')
axes[2].set_title('Income: pd.qcut() Equal-Frequency Boundaries')
axes[2].set_xlabel('Income')
axes[2].legend(fontsize=7.5)
axes[2].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

## 2. Categorical Encoding

ML models require numeric inputs. The choice of encoding depends on whether the category has **meaningful order**.

| Method | When to use |
|---|---|
| **One-Hot Encoding** | Nominal variables — no natural order (city, product type) |
| **Label Encoding** | Simple integer mapping — use carefully for nominal data |
| **Ordinal Encoding** | Variables with natural ordering (Bronze < Silver < Gold) |

In [ ]:
# One-Hot Encoding — city has no meaningful order
city_dummies = pd.get_dummies(df['city'], prefix='city', drop_first=True)
print("One-hot encoded city columns:")
print(city_dummies.head())

In [ ]:
# Ordinal Encoding — tier has a natural hierarchy
tier_order = {'Bronze': 1, 'Silver': 2, 'Gold': 3, 'Platinum': 4}
df['tier_encoded'] = df['tier'].map(tier_order)

print("Ordinal encoding — tier:")
print(df[['tier', 'tier_encoded']].value_counts().sort_index())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Categorical Encoding — Three Methods Side by Side',
             fontsize=13, fontweight='bold')

# Panel 1 — One-Hot Encoding: binary matrix (10-row sample)
sample_ohe = pd.get_dummies(df['city'].head(10), prefix='city', drop_first=True).astype(int)
sns.heatmap(sample_ohe, annot=True, fmt='d', cmap='Blues',
            linewidths=0.8, cbar=False, ax=axes[0],
            annot_kws={'fontsize': 10})
axes[0].set_title('One-Hot Encoding (city)\n✓ No false ordering implied')
axes[0].set_xlabel('Dummy columns (Bangalore = reference)')
axes[0].set_ylabel('Row index')

# Panel 2 — Ordinal Encoding: meaningful hierarchy
tier_map   = {'Bronze': 1, 'Silver': 2, 'Gold': 3, 'Platinum': 4}
tier_cols  = ['#CD7F32', '#A8A9AD', '#FFD700', '#B9F2FF']
bars = axes[1].barh(list(tier_map.keys()), list(tier_map.values()),
                    color=tier_cols, edgecolor='white', linewidth=1.5, height=0.5)
for bar, val in zip(bars, tier_map.values()):
    axes[1].text(val + 0.04, bar.get_y() + bar.get_height() / 2,
                 str(val), va='center', fontsize=13, fontweight='bold')
axes[1].set_xlim(0, 5)
axes[1].set_xlabel('Encoded integer')
axes[1].set_title('Ordinal Encoding (tier)\n✓ Order preserved: Bronze < Silver < Gold < Platinum')
axes[1].spines[['top', 'right']].set_visible(False)

# Panel 3 — Label Encoding: arbitrary integers (danger for nominal data)
cities    = sorted(df['city'].unique())
lbl_vals  = list(range(len(cities)))
bars2 = axes[2].barh(cities, lbl_vals, color='#EF9A9A',
                     edgecolor='white', linewidth=1.5, height=0.5)
for bar, val in zip(bars2, lbl_vals):
    axes[2].text(val + 0.04, bar.get_y() + bar.get_height() / 2,
                 str(val), va='center', fontsize=12, fontweight='bold')
axes[2].set_xlim(0, len(cities) + 0.8)
axes[2].set_xlabel('Encoded integer')
axes[2].set_title('Label Encoding (city)\n⚠ Implies Delhi > Bangalore — meaningless!')
axes[2].set_facecolor('#FFF3E0')
axes[2].text(0.5, -0.18,
             'Use label encoding only for tree models or ordinal variables',
             transform=axes[2].transAxes, ha='center',
             fontsize=8.5, color='#B71C1C', style='italic')
axes[2].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Label Encoding — integer mapping
le = LabelEncoder()
df['city_label'] = le.fit_transform(df['city'])

print("Label encoding mapping:")
for cls, idx in zip(le.classes_, range(len(le.classes_))):
    print(f"  {cls} → {idx}")

## 3. Date / Time Feature Extraction

A timestamp contains far more information than just a date. Extracting temporal components lets models capture **seasonal behaviour**.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Date Feature Extraction — Temporal Patterns Revealed',
             fontsize=13, fontweight='bold')

# Panel 1 — Monthly signups (aggregated across all years)
month_labels = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']
month_counts = df.groupby('month').size().reindex(range(1, 13), fill_value=0)
axes[0].bar(range(1, 13), month_counts.values,
            color='#42A5F5', edgecolor='white', alpha=0.88)
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(month_labels, fontsize=8, rotation=45)
axes[0].set_title('Signups by Month\n(dt.month — seasonal patterns)')
axes[0].set_ylabel('Count')
axes[0].spines[['top', 'right']].set_visible(False)

# Panel 2 — Quarterly distribution
q_counts = df.groupby('quarter').size()
q_colors = ['#1565C0', '#2E7D32', '#E65100', '#6A1B9A']
bars = axes[1].bar(['Q1', 'Q2', 'Q3', 'Q4'], q_counts.values,
                   color=q_colors, edgecolor='white', alpha=0.88)
for bar, val in zip(bars, q_counts.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 1, str(val),
                 ha='center', fontweight='bold')
axes[1].set_title('Signups by Quarter\n(dt.quarter — business reporting periods)')
axes[1].set_ylabel('Count')
axes[1].spines[['top', 'right']].set_visible(False)

# Panel 3 — Weekday vs Weekend stacked bar by quarter
wk_q = df.groupby(['quarter', 'is_weekend']).size().unstack(fill_value=0)
wk_q.columns = ['Weekday', 'Weekend']
wk_q.index   = ['Q1', 'Q2', 'Q3', 'Q4']
wk_q.plot(kind='bar', stacked=True, ax=axes[2],
          color=['#1565C0', '#E65100'], edgecolor='white', alpha=0.88, width=0.5)
axes[2].set_title('Weekday vs Weekend by Quarter\n(is_weekend — behavioural signal)')
axes[2].set_ylabel('Count')
axes[2].set_xlabel('')
axes[2].tick_params(axis='x', rotation=0)
axes[2].legend(title='Day type')
axes[2].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
df['year']        = df['signup_date'].dt.year
df['month']       = df['signup_date'].dt.month
df['day_of_week'] = df['signup_date'].dt.dayofweek   # 0=Monday
df['quarter']     = df['signup_date'].dt.quarter
df['is_weekend']  = (df['signup_date'].dt.dayofweek >= 5).astype(int)
df['hour']        = df['signup_date'].dt.hour
df['is_business_hours'] = df['hour'].between(9, 17).astype(int)

print("Extracted date features:")
df[['signup_date','year','month','day_of_week','quarter','is_weekend','is_business_hours']].head(6)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Interaction Features — Joint Effects Single Variables Cannot Capture',
             fontsize=13, fontweight='bold')

# Panel 1 — Scatter: income_per_age vs profit_margin coloured by tier
sc = axes[0].scatter(df['income_per_age'], df['profit_margin'],
                     c=df['tier_encoded'], cmap='plasma',
                     alpha=0.55, s=12, edgecolors='none')
plt.colorbar(sc, ax=axes[0], label='Tier (1=Bronze … 4=Platinum)')
axes[0].set_xlabel('income_per_age')
axes[0].set_ylabel('profit_margin')
axes[0].set_title('income_per_age vs profit_margin\n(colour = tier — joint effect visible)')
axes[0].spines[['top', 'right']].set_visible(False)

# Panel 2 — Profit margin distribution with mean/median markers
pm = df['profit_margin']
axes[1].hist(pm, bins=40, color='#42A5F5', edgecolor='white', alpha=0.85)
axes[1].axvline(pm.mean(),   color='#e63946', lw=2, ls='--', label=f'Mean   = {pm.mean():.3f}')
axes[1].axvline(pm.median(), color='#2a9d8f', lw=2, ls='--', label=f'Median = {pm.median():.3f}')
axes[1].set_xlabel('profit_margin')
axes[1].set_ylabel('Count')
axes[1].set_title('profit_margin = (price − cost) / price\nInteraction of two raw columns')
axes[1].legend(fontsize=8)
axes[1].spines[['top', 'right']].set_visible(False)

# Panel 3 — Median CLV proxy by loyalty tier
tier_order = ['Bronze', 'Silver', 'Gold', 'Platinum']
clv_by_tier = df.groupby('tier')['clv_proxy'].median().reindex(tier_order)
tier_cols   = ['#CD7F32', '#A8A9AD', '#FFD700', '#B9F2FF']
bars = axes[2].bar(tier_order, clv_by_tier.values,
                   color=tier_cols, edgecolor='white', linewidth=1.5, alpha=0.92)
for bar, val in zip(bars, clv_by_tier.values):
    axes[2].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() * 1.01, f'{val:,.0f}',
                 ha='center', fontsize=9, fontweight='bold')
axes[2].set_title('Median CLV Proxy by Tier\nrevenue × score × tier_encoded')
axes[2].set_ylabel('Median CLV Proxy')
axes[2].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Signups by day of week — reveals weekly cycles
dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
ax = df.groupby('day_of_week').size().plot(kind='bar', figsize=(8, 3),
                                            title='Signups by Day of Week')
ax.set_xticklabels(dow_labels, rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

income = df['income']

transforms = [
    (income,              'Raw Income',          '#e63946'),
    (np.log1p(income),    'log1p(Income)',        '#2a9d8f'),
    (np.sqrt(income),     'sqrt(Income)',         '#3d5a80'),
    (np.cbrt(income),     'cbrt(Income)',         '#6A1B9A'),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Numeric Transformations — Reducing Right Skewness',
             fontsize=13, fontweight='bold')

for ax, (data, title, color) in zip(axes, transforms):
    ax.hist(data, bins=45, color=color, edgecolor='white', alpha=0.85)
    skew_val = data.skew()
    txt_color = '#e63946' if abs(skew_val) > 1 else '#2a9d8f'
    ax.set_title(f'{title}', fontsize=10)
    ax.text(0.97, 0.96, f'skew = {skew_val:.2f}',
            transform=ax.transAxes, ha='right', va='top',
            fontsize=11, fontweight='bold', color=txt_color,
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    ax.spines[['top', 'right']].set_visible(False)

# Annotate the "good" skewness threshold
for ax in axes[1:]:
    ax.annotate('', xy=(0.97, 0.85), xytext=(0.97, 0.93),
                xycoords='axes fraction', textcoords='axes fraction',
                arrowprops=dict(arrowstyle='->', color='#2a9d8f', lw=1.5))

plt.tight_layout()
plt.show()

print("Skewness summary:")
for data, title, _ in transforms:
    print(f"  {title:<22} skew = {data.skew():.3f}")

## 4. Interaction Features

Combine multiple variables to capture **joint effects** that individual variables cannot express alone.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Use income_log as the demonstration feature (already computed)
raw_vals = df['income_log'].dropna()
std_vals = X_std['income_log_std']
mm_vals  = X_mm['income_log_mm']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Feature Scaling — Same Distribution Shape, Different Numeric Range',
             fontsize=13, fontweight='bold')

kw = dict(bins=40, edgecolor='white', alpha=0.85)

# Panel 1 — Raw (income_log)
axes[0].hist(raw_vals, color='#90CAF9', **kw)
axes[0].set_title(f'Raw: income_log\nRange [{raw_vals.min():.1f}, {raw_vals.max():.1f}]')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Count')
axes[0].spines[['top', 'right']].set_visible(False)

# Panel 2 — StandardScaler (z-scores)
axes[1].hist(std_vals, color='#A5D6A7', **kw)
axes[1].axvline(0, color='#e63946', lw=2, ls='--', label='mean = 0')
axes[1].axvline( 1, color='#555', lw=1.2, ls=':', alpha=0.7, label='±1 std')
axes[1].axvline(-1, color='#555', lw=1.2, ls=':', alpha=0.7)
axes[1].set_title(f'StandardScaler\nMean≈{std_vals.mean():.2f}, Std≈{std_vals.std():.2f}')
axes[1].set_xlabel('z-score')
axes[1].legend(fontsize=8)
axes[1].spines[['top', 'right']].set_visible(False)

# Panel 3 — MinMaxScaler ([0, 1])
axes[2].hist(mm_vals, color='#FFCC80', **kw)
axes[2].axvline(0, color='#e63946', lw=2, ls='--', label='min = 0')
axes[2].axvline(1, color='#2a9d8f', lw=2, ls='--', label='max = 1')
axes[2].set_title(f'MinMaxScaler\nRange [{mm_vals.min():.2f}, {mm_vals.max():.2f}]')
axes[2].set_xlabel('Normalised value')
axes[2].legend(fontsize=8)
axes[2].spines[['top', 'right']].set_visible(False)

# Annotation below: shape is preserved
fig.text(0.5, -0.04,
         'Shape of distribution is identical across all three — only the numeric scale changes.',
         ha='center', fontsize=10, style='italic', color='#555')

plt.tight_layout()
plt.show()

In [ ]:
# Multiplicative interactions
df['revenue']        = df['quantity'] * df['price']     # Price × Quantity
df['profit_margin']  = (df['price'] - df['cost']) / df['price']  # Revenue / Cost → Margin
df['income_per_age'] = df['income'] / df['age']         # Normalise income by age

# Business-specific KPI: Customer Lifetime Value proxy
df['clv_proxy'] = df['revenue'] * df['score'] * df['tier_encoded']

print("New interaction features:")
df[['revenue', 'profit_margin', 'income_per_age', 'clv_proxy']].describe().round(2)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler

# Compute scaler parameters fitted two ways: full dataset vs train only
leaky_scaler   = StandardScaler().fit(X_raw)
correct_scaler = StandardScaler().fit(X_train)  # scaler from the cell above

leaky_means   = leaky_scaler.mean_
correct_means = correct_scaler.mean_
leaky_stds    = leaky_scaler.scale_
correct_stds  = correct_scaler.scale_

x, w = np.arange(len(features)), 0.35

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Feature Leakage — Full-Data Scaler vs Train-Only Scaler',
             fontsize=13, fontweight='bold')

# Panel 1 — Learned means
axes[0].bar(x - w/2, leaky_means,   w, color='#e63946', alpha=0.85,
            label='Fit on full dataset (leaky)')
axes[0].bar(x + w/2, correct_means, w, color='#2a9d8f', alpha=0.85,
            label='Fit on train only (correct)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(features, rotation=15, ha='right')
axes[0].set_ylabel('Learned mean')
axes[0].set_title('Means Learned per Feature\n(red ≠ green = test data influenced training scaler)')
axes[0].legend(fontsize=8)
axes[0].spines[['top', 'right']].set_visible(False)

# Panel 2 — Learned std devs
axes[1].bar(x - w/2, leaky_stds,   w, color='#e63946', alpha=0.85,
            label='Fit on full dataset (leaky)')
axes[1].bar(x + w/2, correct_stds, w, color='#2a9d8f', alpha=0.85,
            label='Fit on train only (correct)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(features, rotation=15, ha='right')
axes[1].set_ylabel('Learned std dev')
axes[1].set_title('Std Devs Learned per Feature\n(gap = magnitude of contamination)')
axes[1].legend(fontsize=8)
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

print("Contamination — absolute difference in learned means:")
for feat, lm, cm in zip(features, leaky_means, correct_means):
    print(f"  {feat:<20} Δmean = {abs(lm - cm):.4f}")

## 5. Numeric Transformations

Real-world variables are often skewed. Log transformation compresses large values and reduces skewness, stabilising variance for linear models.

> **Rule**: transform only when necessary, not automatically. Always examine the distribution before and after.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['income'].hist(bins=50, ax=axes[0])
axes[0].set_title(f'Income (skew={df["income"].skew():.2f})')

df['income_log'] = np.log1p(df['income'])
df['income_log'].hist(bins=50, ax=axes[1], color='orange')
axes[1].set_title(f'Log(Income) (skew={df["income_log"].skew():.2f})')

plt.tight_layout()
plt.show()

## 6. Feature Scaling

Scaling ensures numeric variables operate on comparable ranges.

| Method | Formula | When to use |
|---|---|---|
| **Standardisation** | (x - mean) / std → mean=0, std=1 | Distance-based models, neural networks |
| **Normalisation** | (x - min) / (max - min) → [0, 1] | Bounded ranges, CNN inputs |

> Tree-based models (Random Forest, XGBoost) **do not** require scaling.

In [ ]:
num_cols = ['age', 'income_log', 'score', 'revenue']
X = df[num_cols].copy()

# Standardisation (Z-score)
std_scaler = StandardScaler()
X_std = pd.DataFrame(std_scaler.fit_transform(X), columns=[c + '_std' for c in num_cols])

# Normalisation (Min-Max)
mm_scaler = MinMaxScaler()
X_mm = pd.DataFrame(mm_scaler.fit_transform(X), columns=[c + '_mm' for c in num_cols])

print("Standardised (mean≈0, std≈1):")
print(X_std.describe().round(3))

print("\nNormalised (range [0, 1]):")
print(X_mm.describe().round(3))

## 7. Feature Leakage

Leakage leads to **artificially high validation scores but poor real-world performance**.

| Type | Description | Example |
|---|---|---|
| **Target Leakage** | Feature contains future target info | Using "loan approved" to predict "loan default" |
| **Temporal Leakage** | Future data used to predict the past | Next month's sales to predict this month |
| **Data Contamination** | Test set statistics influence training transforms | Fitting scaler on full dataset before split |

> **Golden rule**: if the information would not be available at prediction time, it cannot be used in training.

In [ ]:
from sklearn.model_selection import train_test_split

features = ['age', 'income_log', 'score', 'tier_encoded']
target   = 'revenue'

X_raw = df[features].copy()
y     = df[target].copy()

X_train, X_test, y_train, y_test = train_test_split(X_raw, y, test_size=0.2, random_state=42)

# Correct: fit scaler ONLY on training data, then apply to test
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on train
X_test_scaled  = scaler.transform(X_test)         # transform only on test

print("Correct fit-transform pattern applied.")
print(f"Train shape: {X_train_scaled.shape}, Test shape: {X_test_scaled.shape}")

# Wrong — causes data contamination:
# scaler.fit_transform(X_raw)  # leaks test statistics into training

## 8. Feature Documentation

Professional data science requires a **Feature Dictionary** capturing: name, data type, source, transformation logic, and business meaning.

In [ ]:
feature_dictionary = pd.DataFrame([
    {'feature': 'age_group',      'type': 'categorical', 'source': 'age',
     'transform': 'pd.cut into 4 age brackets', 'business_meaning': 'Customer life stage'},
    {'feature': 'tier_encoded',   'type': 'ordinal int', 'source': 'tier',
     'transform': 'Manual ordinal map Bronze→1 ... Platinum→4', 'business_meaning': 'Loyalty level'},
    {'feature': 'income_log',     'type': 'float',       'source': 'income',
     'transform': 'np.log1p to reduce right skew', 'business_meaning': 'Purchasing power'},
    {'feature': 'profit_margin',  'type': 'float',       'source': 'price, cost',
     'transform': '(price-cost)/price', 'business_meaning': 'Transaction profitability'},
    {'feature': 'is_weekend',     'type': 'binary int',  'source': 'signup_date',
     'transform': 'dayofweek >= 5', 'business_meaning': 'Weekend signup behaviour'},
])

feature_dictionary

## Key Takeaways

- Feature engineering often has **greater impact than model choice**.
- **Binning** aligns data with business logic and captures non-linear relationships.
- Use **ordinal encoding** when order matters; **one-hot** when it doesn't.
- Always apply the **fit-transform principle**: fit on training data only.
- **Feature leakage** silently inflates metrics — validate carefully.
- Document every feature: name, source, transformation, and business meaning.